### 5. Model Architecture (EQCA-Net)
Industry-grade emotion recognition model:
*   **EfficientNet-B0 backbone** (100% frozen ImageNet initialization)
*   **Multi-Scale Feature Aggregation (MSFA)** across 28x28, 14x14, and 7x7 spatial tokens
*   **Emotion-Query Cross-Attention (EQCA)** utilizing 7 explicit class queries
*   **Native Explainability** for high-resolution attention heatmaps

## 1. Environment Setup

In [ ]:
# Install dependencies
!pip install -q torch torchvision timm kaggle scikit-learn matplotlib seaborn tqdm onnx onnxruntime pillow

In [ ]:
import os
import sys
import json
import math
import copy
import random
import warnings
from pathlib import Path
from collections import Counter, OrderedDict
from dataclasses import dataclass, field
from typing import Optional, Tuple, Dict, List

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler

import torchvision.transforms as T
from torchvision.datasets import ImageFolder

import timm
from timm.data.mixup import Mixup

from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, recall_score, accuracy_score
)

warnings.filterwarnings('ignore')

# ── Verify GPU ──
print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

    DEVICE = torch.device('cuda')
else:
    print("⚠️  No GPU found — training will be slow!")
    DEVICE = torch.device('cpu')

print(f"\nUsing device: {DEVICE}")

In [ ]:
import os, json

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)

creds = {"username": "a2ashi", "key": "216c52b21a33770b1072dcdf52f7b107"}

kaggle_json_path = os.path.expanduser("~/.kaggle/kaggle.json")
with open(kaggle_json_path, "w") as f:
    json.dump(creds, f)

os.chmod(kaggle_json_path, 0o600)

print("✅ Kaggle credentials saved!")


## 2. Configuration

All hyperparameters in one place. Change `DATASET` to switch between FER2013, RAF-DB, or AffectNet.

In [ ]:
@dataclass
class Config:
    """Training configuration — all hyperparameters."""
    
    # ── Dataset ──
    datasets: List[str] = field(default_factory=lambda: ["affectnet7"])
    data_dir: str = "./data"           # Root data directory
    num_classes: int = 7
    image_size: int = 224              # Input resolution
    
    # ── Model ──
    backbone: str = "FaceEmoEfficientNet_Tiny"
    drop_rate: float = 0.5             # Classifier dropout
    drop_path_rate: float = 0.1        # Stochastic depth
    
    # ── Training ──
    epochs: int = 60
    batch_size: int = 64               # Per-GPU batch size
    lr: float = 1e-4                   # Peak learning rate
    min_lr: float = 1e-6               # Minimum LR for cosine
    weight_decay: float = 0.05         # AdamW weight decay
    warmup_epochs: int = 5             # LR warmup epochs
    gradient_clip: float = 1.0         # Max gradient norm
    accumulation_steps: int = 1        # Gradient accumulation
    
    # ── Regularization ──
    label_smoothing: float = 0.1       # Label smoothing epsilon
    mixup_alpha: float = 0.4           # MixUp alpha (0 = disabled)
    cutmix_alpha: float = 1.0          # CutMix alpha (0 = disabled)
    mixup_prob: float = 1.0            # Probability of applying mix
    use_class_weights: bool = False     # Inverse-frequency weighting
    
    # ── EMA ──
    use_ema: bool = True               # Exponential Moving Average
    ema_decay: float = 0.9998          # EMA decay rate
    
    # ── Mixed Precision ──
    use_amp: bool = True               # Automatic Mixed Precision
    
    # ── Saving ──
    output_dir: str = "./outputs"      # Checkpoints & logs
    save_best: bool = True
    seed: int = 42
    num_workers: int = 4               # DataLoader workers
    
    # ── Emotion labels ──
    emotion_labels: List[str] = field(default_factory=lambda: [
        'angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise'
    ])


# ── Create config ──
cfg = Config()

# Print config
print("=" * 60)
print("CONFIGURATION")
print("=" * 60)
for k, v in vars(cfg).items():
    print(f"  {k:25s} = {v}")
print("=" * 60)

In [ ]:
# ── Reproducibility ──
def seed_everything(seed: int):
    """Set all random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True  # Keep True for A100 perf

seed_everything(cfg.seed)
os.makedirs(cfg.output_dir, exist_ok=True)
print(f"Seed set to {cfg.seed}")

## 3. Dataset Download

Downloads datasets from Kaggle directly to the supercomputer.  
**Prerequisites**: Place your `kaggle.json` API key in `~/.kaggle/kaggle.json`

```bash
# Get your API key from: https://www.kaggle.com/settings → API → Create New Token
mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
```

In [ ]:
# Cleanup old data directory to prevent mixed datasets
import shutil
import os

if os.path.exists('./data'):
    print("Cleaning up old data directory...")
    try:
        shutil.rmtree('./data')
        print("✅ Cleanup complete.")
    except Exception as e:
        print(f"⚠️ Could not delete directory: {e}")


In [ ]:
import zipfile
import shutil
import os

def download_dataset(dataset_name: str, data_dir: str = "./data"):
    """Download and extract dataset from Kaggle into its own specific folder."""
    kaggle_slugs = {
        "fer2013": "msambare/fer2013",
        "rafdb": "shuvoalok/raf-db-dataset",
        "affectnet7": "lintongdai/affectnet7",
    }
    
    slug = kaggle_slugs.get(dataset_name)
    if slug is None:
        raise ValueError(f"Unknown dataset: {dataset_name}. Choose from {list(kaggle_slugs.keys())}")
    
    # Crucial fix: create a specific directory for THIS dataset
    target_dir = os.path.join(data_dir, dataset_name)
    os.makedirs(target_dir, exist_ok=True)
    
    # Check if already downloaded
    if os.path.exists(target_dir) and len(os.listdir(target_dir)) > 0:
        # Check if it has actual data, not just empty folders
        has_files = any(os.path.isfile(os.path.join(dp, f)) for dp, dn, filenames in os.walk(target_dir) for f in filenames)
        if has_files:
            print(f"✅ Dataset '{dataset_name}' already exists at {target_dir}")
            return target_dir
    
    print(f"📥 Downloading {dataset_name} from Kaggle ({slug}) into {target_dir}...")
    # Extract directly into target_dir
    ret = os.system(f"kaggle datasets download -d {slug} -p {target_dir} --unzip")
    if ret != 0:
        print(f"❌ Failed to download {dataset_name}. Is Kaggle API key configured?")
    
    # Fix nested folders (e.g. if extraction creates target_dir/dataset_name/train)
    for item in os.listdir(target_dir):
        full_path = os.path.join(target_dir, item)
        if os.path.isdir(full_path):
            if any(sub in os.listdir(full_path) for sub in ['train', 'test', 'val']):
                # The actual train/test folders are one level down. Move them up.
                for sub in os.listdir(full_path):
                    shutil.move(os.path.join(full_path, sub), target_dir)
                try:
                    os.rmdir(full_path)
                except OSError:
                    pass # Ignore if not empty
                break
                
    print(f"✅ Dataset ready at: {target_dir}")
    return target_dir


# Download all selected datasets
dataset_paths = [download_dataset(ds, cfg.data_dir) for ds in cfg.datasets]

# Show directory structure
for path in dataset_paths:
    print(f"\n📂 Directory contents of {path}:")
    if not os.path.exists(path): continue
    for item in sorted(os.listdir(path)):
        full = os.path.join(path, item)
        if os.path.isdir(full):
            n_files = sum(len(files) for _, _, files in os.walk(full))
            print(f"  📁 {item}/ ({n_files} files)")
        else:
            print(f"  📄 {item}")

## 4. Data Pipeline

Smart dataset loader that auto-detects the format of FER2013, RAF-DB, and AffectNet.  
Includes proven augmentations for facial expression recognition.

In [ ]:
# ══════════════════════════════════════════════════════════════
# DATA TRANSFORMS — Proven augmentations for FER
# ══════════════════════════════════════════════════════════════

def get_train_transforms(image_size: int = 224):
    """Training augmentations optimized for facial expression recognition."""
    return T.Compose([
        T.Resize((image_size + 32, image_size + 32)),  # Resize slightly larger
        T.RandomCrop(image_size),                       # Random crop to target
        T.RandomHorizontalFlip(p=0.5),                  # Faces are ~symmetric
        T.RandomRotation(degrees=15),                   # Slight rotation
        T.RandomAffine(
            degrees=0,
            translate=(0.1, 0.1),                        # Small translation
            scale=(0.9, 1.1),                            # Slight scale jitter
        ),
        T.ColorJitter(
            brightness=0.3,
            contrast=0.3,
            saturation=0.2,
            hue=0.05,
        ),
        T.RandomGrayscale(p=0.1),                       # Occasional grayscale
        T.RandomErasing(p=0.25, scale=(0.02, 0.15)),    # Simulates occlusion
        T.ToTensor(),
        T.Normalize(
            mean=[0.485, 0.456, 0.406],                 # ImageNet stats
            std=[0.229, 0.224, 0.225],
        ),
    ])


def get_val_transforms(image_size: int = 224):
    """Deterministic validation transforms."""
    return T.Compose([
        T.Resize((image_size, image_size)),
        T.ToTensor(),
        T.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ])


# Note: RandomErasing is applied after ToTensor, so reorder
def get_train_transforms_v2(image_size: int = 224):
    """Corrected training transforms with RandomErasing after ToTensor."""
    return T.Compose([
        T.Resize((image_size + 32, image_size + 32)),
        T.RandomCrop(image_size),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomRotation(degrees=15),
        T.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
        T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
        T.RandomGrayscale(p=0.1),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        T.RandomErasing(p=0.25, scale=(0.02, 0.15)),  # After ToTensor
    ])

print("✅ Transforms defined")

In [ ]:
# ══════════════════════════════════════════════════════════════
# DATASET LOADING — Auto-detect format for all 3 datasets
# ══════════════════════════════════════════════════════════════

def find_image_dirs(root: str) -> Tuple[str, str]:
    """
    Auto-detect train and test/val directories.
    Supports:
      - fer2013:    root/train/  root/test/
      - rafdb:      root/train/  root/test/
      - affectnet7: root/train/  root/val/ (or root/test/)
    """
    root = Path(root)
    
    # Direct structure: root/train/ root/test/
    train_candidates = ['train', 'training', 'Train']
    val_candidates = ['test', 'val', 'validation', 'Test', 'Val']
    
    train_dir = None
    val_dir = None
    
    # Search direct children
    for item in root.iterdir():
        if item.is_dir():
            if item.name in train_candidates:
                train_dir = str(item)
            elif item.name in val_candidates:
                val_dir = str(item)
    
    # If not found, search one level deeper
    if train_dir is None:
        for sub in root.iterdir():
            if sub.is_dir():
                for item in sub.iterdir():
                    if item.is_dir():
                        if item.name in train_candidates:
                            train_dir = str(item)
                        elif item.name in val_candidates:
                            val_dir = str(item)
    
    if train_dir is None:
        raise FileNotFoundError(
            f"Could not find train directory in {root}. "
            f"Contents: {[x.name for x in root.iterdir()]}"
        )
    if val_dir is None:
        raise FileNotFoundError(
            f"Could not find test/val directory in {root}. "
            f"Contents: {[x.name for x in root.iterdir()]}"
        )
    
    return train_dir, val_dir


def build_label_mapping(train_dir: str) -> Dict[str, int]:
    """
    Build a consistent label mapping from folder names → indices.
    Maps to the standard 7-class FER order.
    """
    # Standard order
    standard_order = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
    
    # Get folder names
    folder_names = sorted([
        d.name.lower() for d in Path(train_dir).iterdir() 
        if d.is_dir() and not d.name.startswith('.')
    ])
    
    print(f"  Found class folders: {folder_names}")
    
    # Try to match to standard order
    label_map = {}
    for folder in folder_names:
        folder_lower = folder.lower()
        if folder_lower in standard_order:
            label_map[folder] = standard_order.index(folder_lower)
        else:
            # Fallback: use alphabetical index
            label_map[folder] = len(label_map)
    
    return label_map


# ── Load datasets ──
from torch.utils.data import ConcatDataset, Dataset
from PIL import Image

class UnifiedFERDataset(Dataset):
    def __init__(self, root, transform=None):
        self.root = root
        self.transform = transform
        self.classes = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
        self.samples = []
        
        if not os.path.exists(root):
            return
            
        # AffectNet standard 7-class mapping if folders are numeric
        numeric_mapping = {
            '0': 4, # Neutral
            '1': 3, # Happy
            '2': 5, # Sad
            '3': 6, # Surprise
            '4': 2, # Fear
            '5': 1, # Disgust
            '6': 0, # Anger
        }
            
        for cls_folder in os.listdir(root):
            cls_path = os.path.join(root, cls_folder)
            if not os.path.isdir(cls_path): continue
            
            cls_lower = cls_folder.lower()
            
            if cls_lower in numeric_mapping:
                idx = numeric_mapping[cls_lower]
            elif cls_lower in self.classes:
                idx = self.classes.index(cls_lower)
            elif 'anger' in cls_lower: idx = 0
            elif 'happiness' in cls_lower: idx = 3
            elif 'sadness' in cls_lower: idx = 5
            else:
                continue # Skip unmapped classes
                
            for img_name in os.listdir(cls_path):
                if img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                    self.samples.append((os.path.join(cls_path, img_name), idx))
                    
    def __len__(self):
        return len(self.samples)
        
    def __getitem__(self, i):
        path, label = self.samples[i]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

train_datasets, val_datasets = [], []

for ds_name, ds_path in zip(cfg.datasets, dataset_paths):
    print(f"\nLoading dataset: {ds_name}")
    try:
        train_dir, val_dir = find_image_dirs(ds_path)
    except Exception as e:
        print(f"  Skipping {ds_name} - Not properly downloaded or extracted. ({e})")
        continue
        
    print(f"  Train dir: {train_dir}")
    print(f"  Val dir:   {val_dir}")

    t_ds = UnifiedFERDataset(root=train_dir, transform=get_train_transforms_v2(cfg.image_size))
    v_ds = UnifiedFERDataset(root=val_dir, transform=get_val_transforms(cfg.image_size))
    
    print(f"  Found {len(t_ds)} train, {len(v_ds)} val images.")
    if len(t_ds) > 0: train_datasets.append(t_ds)
    if len(v_ds) > 0: val_datasets.append(v_ds)

# Combine datasets
train_dataset = ConcatDataset(train_datasets)
val_dataset = ConcatDataset(val_datasets)

# Standardize config class metadata
cfg.num_classes = 7
cfg.emotion_labels = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']

print(f"\n📊 Dataset Statistics:")
print(f"  Classes ({cfg.num_classes}): {cfg.emotion_labels}")
print(f"  Total Train samples: {len(train_dataset):,}")
print(f"  Total Val samples:   {len(val_dataset):,}")

# Class distribution
train_targets = []
for ds in train_datasets:
    train_targets.extend([s[1] for s in ds.samples])
    
class_counts = Counter(train_targets)
print(f"\n  Combined Class distribution (train):")
for cls_idx in sorted(class_counts.keys()):
    cls_name = cfg.emotion_labels[cls_idx]
    count = class_counts[cls_idx]
    pct = 100 * count / len(train_targets) if len(train_targets) > 0 else 0
    bar = '█' * int(pct)
    print(f"    {cls_name:12s}: {count:6d} ({pct:5.1f}%) {bar}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# DATALOADERS — With class-balanced sampling
# ══════════════════════════════════════════════════════════════

def compute_class_weights(targets: List[int], num_classes: int) -> torch.Tensor:
    """Compute inverse-frequency class weights."""
    counts = Counter(targets)
    total = sum(counts.values())
    weights = torch.zeros(num_classes)
    for cls_idx, count in counts.items():
        weights[cls_idx] = total / (num_classes * count)
    return weights


def create_weighted_sampler(targets: List[int]) -> WeightedRandomSampler:
    """Create a weighted random sampler for class-balanced batches."""
    counts = Counter(targets)
    class_weights = {cls: 1.0 / count for cls, count in counts.items()}
    sample_weights = [class_weights[t] for t in targets]
    return WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(targets),
        replacement=True,
    )


# Class weights for loss function
class_weights = compute_class_weights(train_targets, cfg.num_classes).to(DEVICE)
print(f"Class weights: {class_weights.cpu().numpy().round(3)}")

# Weighted sampler for balanced batches
train_sampler = create_weighted_sampler(train_targets)

# DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.batch_size,
    sampler=train_sampler,
    num_workers=cfg.num_workers,
    pin_memory=True,
    drop_last=True,
    persistent_workers=True if cfg.num_workers > 0 else False,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=cfg.batch_size * 2,  # Larger batch for eval (no grads)
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=True,
    persistent_workers=True if cfg.num_workers > 0 else False,
)

print(f"\n✅ DataLoaders created")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches:   {len(val_loader)}")

In [ ]:
# ── Visualize a batch ──
def show_batch(loader, classes, n=16):
    """Visualize a batch of training images."""
    images, labels = next(iter(loader))
    
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    
    fig, axes = plt.subplots(2, n // 2, figsize=(20, 6))
    for i, ax in enumerate(axes.flat):
        if i >= min(n, len(images)):
            break
        img = images[i] * std + mean  # Denormalize
        img = img.clamp(0, 1).permute(1, 2, 0).numpy()
        ax.imshow(img)
        ax.set_title(classes[labels[i]], fontsize=10)
        ax.axis('off')
    plt.suptitle('Training Batch Sample', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

show_batch(train_loader, cfg.emotion_labels)

### 5. Model Architecture (EQCA-Net)
Industry-grade emotion recognition model:
*   **EfficientNet-B0 backbone** (100% frozen ImageNet initialization)
*   **Multi-Scale Feature Aggregation (MSFA)** across 28x28, 14x14, and 7x7 spatial tokens
*   **Emotion-Query Cross-Attention (EQCA)** utilizing 7 explicit class queries
*   **Native Explainability** for high-resolution attention heatmaps

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm

class EmotionQueryAttention(nn.Module):
    """Cross-Attention using Learnable Emotion Queries"""
    def __init__(self, feature_dim, num_emotions=7, dropout=0.2):
        super().__init__()
        self.num_emotions = num_emotions
        self.feature_dim = feature_dim
        
        # Learnable emotion queries with proper scaled initialization
        self.emotion_queries = nn.Parameter(torch.randn(1, num_emotions, feature_dim) / math.sqrt(feature_dim))
        
        self.multihead_attn = nn.MultiheadAttention(embed_dim=feature_dim, num_heads=8, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(feature_dim)
        self.norm2 = nn.LayerNorm(feature_dim)
        self.dropout = nn.Dropout(dropout)
        
        # Feed Forward Network
        self.ffn = nn.Sequential(
            nn.Linear(feature_dim, feature_dim * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(feature_dim * 4, feature_dim),
            nn.Dropout(dropout)
        )
        
    def forward(self, feature_maps):
        B = feature_maps.size(0)
        
        # queries shape: (B, num_emotions, feature_dim)
        queries = self.emotion_queries.expand(B, -1, -1)
        
        # Cross attention
        attn_out, attn_weights = self.multihead_attn(query=queries, key=feature_maps, value=feature_maps)
        
        out = self.norm1(queries + self.dropout(attn_out))
        ffn_out = self.ffn(out)
        out = self.norm2(out + ffn_out)
        
        return out, attn_weights

class FocalLoss(nn.Module):
    """Focal Loss with Label Smoothing to handle class imbalance"""
    def __init__(self, alpha=1, gamma=2, label_smoothing=0.1):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, label_smoothing=self.label_smoothing, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()

class FaceEmoEfficientNet(nn.Module):
    def __init__(
        self,
        model_name="efficientnet_b0",
        num_classes=7,
        pretrained=True
    ):
        super().__init__()
        self.num_classes = num_classes

        # Load timm EfficientNet and extract multi-scale features
        self.backbone = timm.create_model(model_name, pretrained=pretrained, features_only=True)
        
        # Determine feature dimensions based on model choice
        # For efficientnet_b0: [16, 24, 40, 112, 320]
        feature_info = self.backbone.feature_info
        channels = [f['num_chs'] for f in feature_info]
        
        self.feature_dim = 256 # Target dimension for MSFA
        
        # We will extract the last 3 stages: indexes 2, 3, 4
        # b0 dimensions: channels[2]=40, channels[3]=112, channels[4]=320
        self.proj2 = nn.Conv2d(channels[2], self.feature_dim, 1)
        self.proj3 = nn.Conv2d(channels[3], self.feature_dim, 1)
        self.proj4 = nn.Conv2d(channels[4], self.feature_dim, 1)
        
        self.norm = nn.LayerNorm(self.feature_dim, eps=1e-6)
        
        # Total spatial tokens: for 224x224 input
        # Stage 2 (28x28) = 784
        # Stage 3 (14x14) = 196
        # Stage 4 (7x7) = 49
        # Total = 1029
        self.pos_embed = nn.Parameter(torch.randn(1, 1029, self.feature_dim) * .02)
        
        # Cross-Attention with Learnable Queries
        self.attention = EmotionQueryAttention(feature_dim=self.feature_dim, num_emotions=num_classes)
        
        # Classifier over all emotion queries
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(num_classes * self.feature_dim, self.feature_dim),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(self.feature_dim, num_classes)
        )

        self.focal_loss = FocalLoss()

        # Initialize the projection layers and head
        self._init_weights()

    def _init_weights(self):
        for m in [self.proj2, self.proj3, self.proj4, self.head]:
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x, labels=None):
        features = self.backbone(x)
        
        # features[2]: (B, 40, 28, 28)
        # features[3]: (B, 112, 14, 14)
        # features[4]: (B, 320, 7, 7)
        
        f2 = self.proj2(features[2]).flatten(2).transpose(1, 2) # (B, 784, dim)
        f3 = self.proj3(features[3]).flatten(2).transpose(1, 2) # (B, 196, dim)
        f4 = self.proj4(features[4]).flatten(2).transpose(1, 2) # (B, 49, dim)
        
        # Combine all features across spatial scales
        multi_scale_features = torch.cat([f2, f3, f4], dim=1) # (B, 1029, dim)
        multi_scale_features = multi_scale_features + self.pos_embed
        multi_scale_features = self.norm(multi_scale_features)
        
        # Emotion Query Attention
        query_out, attn_weights = self.attention(multi_scale_features)
        
        # Score the concatenated queries
        logits = self.head(query_out)

        if labels is not None:
            # Regularization to make emotion queries distinct
            mean_attention_weight = torch.mean(attn_weights)
            attention_loss = torch.mean((attn_weights - mean_attention_weight) ** 2)

            loss = self.focal_loss(logits, labels) + attention_loss
            return logits

        return logits

    def extract_attention_maps(self, x):
        features = self.backbone(x)
        
        f2 = self.proj2(features[2]).flatten(2).transpose(1, 2)
        f3 = self.proj3(features[3]).flatten(2).transpose(1, 2)
        f4 = self.proj4(features[4]).flatten(2).transpose(1, 2)
        
        multi_scale_features = torch.cat([f2, f3, f4], dim=1)
        multi_scale_features = multi_scale_features + self.pos_embed
        multi_scale_features = self.norm(multi_scale_features)
        
        query_out, attn_weights = self.attention(multi_scale_features)
        logits = self.head(query_out)
        
        shapes = [
            (features[2].size(2), features[2].size(3)),
            (features[3].size(2), features[3].size(3)),
            (features[4].size(2), features[4].size(3))
        ]
        
        return logits, attn_weights, shapes

def get_face_emoefficientnet(num_classes=7, model_name="efficientnet_b0"):
    return FaceEmoEfficientNet(model_name=model_name, num_classes=num_classes)


# Initialize the model and move to device
model = get_face_emoefficientnet(num_classes=cfg.num_classes).to(DEVICE)
print("FaceEmoEfficientNet initialized successfully.")


In [ ]:
# ══════════════════════════════════════════════════════════════
# OPTIMIZER, LOSS, SCHEDULER
# ══════════════════════════════════════════════════════════════

backbone_params = []
head_params = []
for name, param in model.named_parameters():
    if 'head' in name or 'attention' in name or 'proj' in name or 'pos_embed' in name or 'norm' in name:
        head_params.append(param)
    else:
        backbone_params.append(param)

# ── Optimizer: AdamW with layer-wise LR decay ──
param_groups = [
    {
        'params': backbone_params,
        'lr': cfg.lr * 0.1,           
        'weight_decay': cfg.weight_decay,
    },
    {
        'params': head_params,
        'lr': cfg.lr,                 
        'weight_decay': cfg.weight_decay * 0.1,  
    },
]

import torch
import torch.optim as optim
optimizer = optim.AdamW(param_groups, betas=(0.9, 0.999))

# ── Loss function ──
import torch.nn as nn
criterion = nn.CrossEntropyLoss(
    weight=class_weights if getattr(cfg, 'use_class_weights', False) else None,
    label_smoothing=getattr(cfg, 'label_smoothing', 0.1),
)

# ── LR Scheduler ──
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=[cfg.lr * 0.1, cfg.lr],
    epochs=cfg.epochs,
    steps_per_epoch=len(train_loader),
    pct_start=5.0 / max(1, cfg.epochs),
    div_factor=25,
    final_div_factor=100
)

# ── Mixed Precision ──
from torch.cuda.amp import GradScaler
scaler = GradScaler(enabled=getattr(cfg, 'use_amp', True))

# ── MixUp / CutMix ──
mixup_fn = None
if getattr(cfg, 'mixup_alpha', 0) > 0 or getattr(cfg, 'cutmix_alpha', 0) > 0:
    from timm.data.mixup import Mixup
    from timm.loss import SoftTargetCrossEntropy
    mixup_fn = Mixup(
        mixup_alpha=getattr(cfg, 'mixup_alpha', 0),
        cutmix_alpha=getattr(cfg, 'cutmix_alpha', 0),
        prob=getattr(cfg, 'mixup_prob', 0.5),
        switch_prob=0.5,
        mode='batch',
        label_smoothing=getattr(cfg, 'label_smoothing', 0.1),
        num_classes=getattr(cfg, 'num_classes', 7),
    )
    mixup_criterion = SoftTargetCrossEntropy()

print("✅ Optimizer, loss, scheduler configured")

In [ ]:

import torch
import torch.nn as nn
from torch.cuda.amp import autocast, GradScaler

model = get_face_emoefficientnet(num_classes=cfg.num_classes).to(DEVICE)
model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()
scaler = GradScaler(enabled=True)

for i in range(5):
    x = torch.randn(8, 3, 224, 224).to(DEVICE)
    y = torch.randint(0, cfg.num_classes, (8,)).to(DEVICE)
    
    optimizer.zero_grad()
    
    with autocast(enabled=True):
        logits = model(x)
        if torch.isnan(logits).any():
            print(f'Iteration {i}: LOGITS ARE NAN!')
            break
        loss = criterion(logits, y)
        
    scaler.scale(loss).backward()
    
    # Check if gradients have NaN before unscale
    has_nan_grad = any(p.grad is not None and torch.isnan(p.grad).any() for p in model.parameters())
    print(f'Iteration {i}: Gradients have NaN before unscale: {has_nan_grad}')
    
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()
    
    # Check if weights have NaN after step
    has_nan_weight = any(torch.isnan(p).any() for p in model.parameters())
    print(f'Iteration {i}: Weights have NaN after step: {has_nan_weight}')


## 6. Training Loop

Full-featured training with:
- Mixed precision (AMP) for 2× speedup on A100
- Cosine annealing with linear warmup
- MixUp + CutMix augmentation
- Gradient clipping
- EMA weight averaging
- Early stopping
- Best model checkpointing

In [ ]:
# ══════════════════════════════════════════════════════════════
# LEARNING RATE SCHEDULER — Cosine with Linear Warmup
# ══════════════════════════════════════════════════════════════

class CosineWarmupScheduler:
    """Cosine annealing with linear warmup."""
    
    def __init__(
        self,
        optimizer: optim.Optimizer,
        warmup_epochs: int,
        total_epochs: int,
        min_lr: float = 1e-6,
        steps_per_epoch: int = 1,
    ):
        self.optimizer = optimizer
        self.warmup_steps = warmup_epochs * steps_per_epoch
        self.total_steps = total_epochs * steps_per_epoch
        self.min_lr = min_lr
        self.base_lrs = [pg['lr'] for pg in optimizer.param_groups]
        self.current_step = 0
    
    def step(self):
        self.current_step += 1
        
        if self.current_step <= self.warmup_steps:
            # Linear warmup
            scale = self.current_step / max(1, self.warmup_steps)
        else:
            # Cosine annealing
            progress = (self.current_step - self.warmup_steps) / max(
                1, self.total_steps - self.warmup_steps
            )
            scale = 0.5 * (1.0 + math.cos(math.pi * progress))
        
        for pg, base_lr in zip(self.optimizer.param_groups, self.base_lrs):
            pg['lr'] = max(self.min_lr, base_lr * scale)
    
    def get_lr(self) -> float:
        return self.optimizer.param_groups[0]['lr']


# ══════════════════════════════════════════════════════════════
# TRAINING UTILITIES
# ══════════════════════════════════════════════════════════════

class AverageMeter:
    """Tracks running average of a metric."""
    def __init__(self):
        self.reset()
    
    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0
    
    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count


print("✅ Training utilities defined")

In [ ]:
# ══════════════════════════════════════════════════════════════
# TRAIN ONE EPOCH
# ══════════════════════════════════════════════════════════════

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """Evaluate model on validation set."""
    model.eval()
    loss_meter = AverageMeter()
    all_preds = []
    all_targets = []
    
    for images, targets in loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        
        with autocast(enabled=cfg.use_amp):
            logits = model(images)
            loss = criterion(logits, targets)
        
        loss_meter.update(loss.item(), images.size(0))
        preds = logits.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(targets.cpu().numpy())
    
    all_preds = np.array(all_preds)
    all_targets = np.array(all_targets)
    
    acc = accuracy_score(all_targets, all_preds)
    uf1 = f1_score(all_targets, all_preds, average='macro')
    uar = recall_score(all_targets, all_preds, average='macro')
    
    return {
        'loss': loss_meter.avg,
        'accuracy': acc,
        'uf1': uf1,
        'uar': uar,
        'predictions': all_preds,
        'targets': all_targets,
    }


def train_one_epoch(
    model, loader, criterion, optimizer, scheduler, scaler,
    mixup_fn, ema, device, epoch, cfg
):
    """Train for one epoch."""
    model.train()
    loss_meter = AverageMeter()
    correct = 0
    total = 0
    
    pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{cfg.epochs}", leave=False)
    
    optimizer.zero_grad()
    
    for step, (images, targets) in enumerate(pbar):
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        
        # Apply MixUp / CutMix
        use_mixup = mixup_fn is not None and random.random() < cfg.mixup_prob
        if use_mixup:
            images, targets_mixed = mixup_fn(images, targets)
        
        # Forward pass with AMP
        with autocast(enabled=cfg.use_amp):
            logits = model(images)
            if use_mixup:
                loss = mixup_criterion(logits, targets_mixed)
            else:
                loss = criterion(logits, targets)
            
            loss = loss / cfg.accumulation_steps
        
        # Backward pass
        scaler.scale(loss).backward()
        
        # Gradient accumulation step
        if (step + 1) % cfg.accumulation_steps == 0:
            # Gradient clipping
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.gradient_clip)
            
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            
            # Update EMA
            if ema is not None:
                ema.update(model)
        
        # LR step (per-iteration)
        scheduler.step()
        
        # Metrics
        loss_meter.update(loss.item() * cfg.accumulation_steps, images.size(0))
        preds = logits.argmax(dim=1)
        if use_mixup:
            dominant_targets = targets_mixed.argmax(dim=1)
            correct += (preds == dominant_targets).sum().item()
        else:
            correct += (preds == targets).sum().item()
        total += targets.size(0)
        
        # Update progress bar
        pbar.set_postfix({
            'loss': f'{loss_meter.avg:.4f}',
            'acc': f'{correct/max(1,total):.3f}',
            'lr': '{:.2e}'.format(optimizer.param_groups[0]['lr']),
        })
    
    return {
        'loss': loss_meter.avg,
        'accuracy': correct / max(1, total),
    }

print("✅ Training functions defined")

In [ ]:

from timm.utils import ModelEmaV2
ema = ModelEmaV2(model, decay=getattr(cfg, 'ema_decay', 0.9998)) if getattr(cfg, 'use_ema', True) else None
print('EMA Initialized:', ema is not None)

# ══════════════════════════════════════════════════════════════
# 🚀 MAIN TRAINING LOOP
# ══════════════════════════════════════════════════════════════

print("=" * 70)
print(f"🚀 STARTING TRAINING")
# print(f"   Dataset:    {", ".join(cfg.datasets)}")
print(f"   Model:      {cfg.backbone}")
print(f"   Epochs:     {cfg.epochs}")
print(f"   Batch size: {cfg.batch_size}")
print(f"   Device:     {DEVICE}")
print("=" * 70)

best_acc = 0.0
best_uf1 = 0.0
best_epoch = 0
patience_counter = 0
patience_limit = 15  # Early stopping patience

history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': [], 'val_uf1': [], 'val_uar': [],
    'lr': [],
}

for epoch in range(cfg.epochs):
    # ── Train ──
    train_metrics = train_one_epoch(
        model, train_loader, criterion, optimizer, scheduler,
        scaler, mixup_fn, ema, DEVICE, epoch, cfg
    )
    
    # ── Evaluate (use EMA model if available) ──
    eval_model = ema.module if ema is not None else model
    val_metrics = evaluate(eval_model, val_loader, criterion, DEVICE)
    
    # ── Log metrics ──
    history['train_loss'].append(train_metrics['loss'])
    history['train_acc'].append(train_metrics['accuracy'])
    history['val_loss'].append(val_metrics['loss'])
    history['val_acc'].append(val_metrics['accuracy'])
    history['val_uf1'].append(val_metrics['uf1'])
    history['val_uar'].append(val_metrics['uar'])
    history['lr'].append(optimizer.param_groups[-1]['lr'])
    
    # ── Print epoch summary ──
    improved = ''
    if val_metrics['accuracy'] > best_acc:
        best_acc = val_metrics['accuracy']
        best_uf1 = val_metrics['uf1']
        best_epoch = epoch + 1
        patience_counter = 0
        improved = ' ⭐ NEW BEST'
        
        # Save best model
        if cfg.save_best:
            save_dict = {
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'ema_state_dict': ema.state_dict() if ema else None,
                'optimizer_state_dict': optimizer.state_dict(),
                'best_acc': best_acc,
                'best_uf1': best_uf1,
                'config': vars(cfg),
                'emotion_labels': cfg.emotion_labels,
            }
            torch.save(save_dict, os.path.join(cfg.output_dir, 'best_model.pth'))
    else:
        patience_counter += 1
    
    print(
        f"Epoch {epoch+1:3d}/{cfg.epochs} │ "
        f"Train Loss: {train_metrics['loss']:.4f}  Acc: {train_metrics['accuracy']:.3f} │ "
        f"Val Loss: {val_metrics['loss']:.4f}  Acc: {val_metrics['accuracy']:.3f}  "
        f"UF1: {val_metrics['uf1']:.3f}  UAR: {val_metrics['uar']:.3f} │ "
        f"LR: {optimizer.param_groups[-1]['lr']:.2e}{improved}"
    )
    
    # ── Early stopping ──
    if patience_counter >= patience_limit:
        print(f"\n⏹️  Early stopping at epoch {epoch+1} (no improvement for {patience_limit} epochs)")
        break

print(f"\n{'='*70}")
print(f"🏆 TRAINING COMPLETE")
print(f"   Best Accuracy: {best_acc:.4f} (epoch {best_epoch})")
print(f"   Best UF1:      {best_uf1:.4f}")
print(f"   Model saved:   {os.path.join(cfg.output_dir, 'best_model.pth')}")
print(f"{'='*70}")

## 7. Training Curves

In [ ]:
# ══════════════════════════════════════════════════════════════
# TRAINING CURVES
# ══════════════════════════════════════════════════════════════

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Loss
axes[0, 0].plot(history['train_loss'], label='Train', linewidth=2)
axes[0, 0].plot(history['val_loss'], label='Val', linewidth=2)
axes[0, 0].set_title('Loss', fontsize=13, fontweight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Accuracy
axes[0, 1].plot(history['train_acc'], label='Train', linewidth=2)
axes[0, 1].plot(history['val_acc'], label='Val', linewidth=2)
axes[0, 1].axhline(y=best_acc, color='r', linestyle='--', alpha=0.5, label=f'Best: {best_acc:.3f}')
axes[0, 1].set_title('Accuracy', fontsize=13, fontweight='bold')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# UF1 / UAR
axes[1, 0].plot(history['val_uf1'], label='UF1 (Macro F1)', linewidth=2, color='green')
axes[1, 0].plot(history['val_uar'], label='UAR (Macro Recall)', linewidth=2, color='orange')
axes[1, 0].set_title('UF1 & UAR', fontsize=13, fontweight='bold')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Learning Rate
axes[1, 1].plot(history['lr'], linewidth=2, color='purple')
axes[1, 1].set_title('Learning Rate', fontsize=13, fontweight='bold')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_yscale('log')
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle(f'Training Report — {cfg.backbone} on {", ".join(cfg.datasets)}', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(cfg.output_dir, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"📊 Training curves saved to {cfg.output_dir}/training_curves.png")

## 8. Evaluation — Detailed Results

In [ ]:
# ══════════════════════════════════════════════════════════════
# LOAD BEST MODEL & EVALUATE
# ══════════════════════════════════════════════════════════════

# Load best checkpoint
ckpt_path = os.path.join(cfg.output_dir, 'best_model.pth')
checkpoint = torch.load(ckpt_path, map_location=DEVICE)

# Use EMA weights if available
if checkpoint.get('ema_state_dict') is not None:
    ema_state_dict = {k.replace('module.', ''): v for k, v in checkpoint['ema_state_dict'].items()}
    model.load_state_dict(ema_state_dict)
    print("✅ Loaded EMA weights (best model)")
else:
    model.load_state_dict(checkpoint['model_state_dict'])
    print("✅ Loaded best model weights")

print(f"   From epoch: {checkpoint['epoch']}")
print(f"   Best accuracy: {checkpoint['best_acc']:.4f}")

# Final evaluation
final_metrics = evaluate(model, val_loader, criterion, DEVICE)

In [ ]:
# ══════════════════════════════════════════════════════════════
# DETAILED CLASSIFICATION REPORT
# ══════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("📋 CLASSIFICATION REPORT")
print("=" * 70)
print(classification_report(
    final_metrics['targets'],
    final_metrics['predictions'],
    target_names=cfg.emotion_labels,
    digits=4,
))

print(f"\n  Overall Accuracy: {final_metrics['accuracy']:.4f}")
print(f"  Macro F1 (UF1):  {final_metrics['uf1']:.4f}")
print(f"  Macro Recall (UAR): {final_metrics['uar']:.4f}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CONFUSION MATRIX
# ══════════════════════════════════════════════════════════════

cm = confusion_matrix(final_metrics['targets'], final_metrics['predictions'])
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Raw counts
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=cfg.emotion_labels,
    yticklabels=cfg.emotion_labels,
    ax=axes[0],
)
axes[0].set_title('Confusion Matrix (Counts)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# Normalized
sns.heatmap(
    cm_normalized, annot=True, fmt='.2f', cmap='Oranges',
    xticklabels=cfg.emotion_labels,
    yticklabels=cfg.emotion_labels,
    ax=axes[1],
)
axes[1].set_title('Confusion Matrix (Normalized)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

plt.suptitle(f'{cfg.backbone} on {", ".join(cfg.datasets)} — Acc: {final_metrics["accuracy"]:.3f}', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(cfg.output_dir, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"📊 Confusion matrix saved to {cfg.output_dir}/confusion_matrix.png")

## 9. Export to ONNX

Export the trained model for production deployment.  
ONNX runs on CPU/GPU with ONNX Runtime — no PyTorch needed at inference.

In [ ]:
# ══════════════════════════════════════════════════════════════
# ONNX EXPORT
# ══════════════════════════════════════════════════════════════

model.eval()
model.cpu()

onnx_path = os.path.join(cfg.output_dir, 'emotion_model.onnx')

# Dummy input
dummy_input = torch.randn(1, 3, cfg.image_size, cfg.image_size)

# Export
torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['logits'],
    dynamic_axes={
        'input': {0: 'batch_size'},
        'logits': {0: 'batch_size'},
    },
)

# Verify ONNX model
import onnx
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)

# Test with ONNX Runtime
import onnxruntime as ort

session = ort.InferenceSession(onnx_path)
ort_input = {session.get_inputs()[0].name: dummy_input.numpy()}
ort_output = session.run(None, ort_input)

# Compare PyTorch vs ONNX output
with torch.no_grad():
    pt_output = model(dummy_input).numpy()

max_diff = np.abs(pt_output - ort_output[0]).max()

onnx_size_mb = os.path.getsize(onnx_path) / (1024 * 1024)

print(f"✅ ONNX model exported successfully!")
print(f"   Path:     {onnx_path}")
print(f"   Size:     {onnx_size_mb:.1f} MB")
print(f"   Max diff: {max_diff:.8f} (PyTorch vs ONNX)")
print(f"   Status:   {'✅ Verified' if max_diff < 1e-4 else '⚠️ Check accuracy'}")

# Move model back to GPU
model.to(DEVICE)
print(f"\n📦 Files in {cfg.output_dir}:")
for f in sorted(os.listdir(cfg.output_dir)):
    size = os.path.getsize(os.path.join(cfg.output_dir, f)) / (1024 * 1024)
    print(f"   {f:35s} {size:8.1f} MB")

## 10. Save Final Results & Metadata

In [ ]:
# ══════════════════════════════════════════════════════════════
# SAVE RESULTS SUMMARY
# ══════════════════════════════════════════════════════════════
import json

# Calculate total parameters
total_params = sum(p.numel() for p in model.parameters())

results = {
    'model': {
        'backbone': getattr(cfg, 'backbone', 'FaceEmoEfficientNet'),
        'num_classes': cfg.num_classes,
        'emotion_labels': cfg.emotion_labels,
        'total_params': total_params,
        'image_size': cfg.image_size,
    },
    'dataset': {
        'name': cfg.datasets,
        'train_samples': len(train_dataset),
        'val_samples': len(val_dataset),
    },
    'training': {
        'epochs_trained': len(history['train_loss']),
        'best_epoch': best_epoch,
        'batch_size': cfg.batch_size,
        'lr': cfg.lr,
        'label_smoothing': getattr(cfg, 'label_smoothing', 0.1),
        'mixup': getattr(cfg, 'mixup_alpha', 0) > 0,
        'cutmix': getattr(cfg, 'cutmix_alpha', 0) > 0,
        'ema': getattr(cfg, 'use_ema', False),
        'amp': getattr(cfg, 'use_amp', True),
    },
    'results': {
        'accuracy': float(final_metrics['accuracy']),
        'uf1_macro_f1': float(final_metrics['uf1']),
        'uar_macro_recall': float(final_metrics['uar']),
        'val_loss': float(final_metrics['loss']),
    },
    'files': {
        'checkpoint': 'best_model.pth',
        'onnx': 'emotion_model.onnx',
        'training_curves': 'training_curves.png',
        'confusion_matrix': 'confusion_matrix.png',
    },
}

results_path = os.path.join(cfg.output_dir, 'results.json')
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f"📋 Results saved to {results_path}")
print(json.dumps(results, indent=2))


## 11. Quick Inference Demo

Test the trained model on sample images.

In [ ]:
# ══════════════════════════════════════════════════════════════
# INFERENCE DEMO
# ══════════════════════════════════════════════════════════════

from PIL import Image

def predict_emotion(model, image_path: str, device, labels, image_size=224):
    """Predict emotion from a single image."""
    transform = get_val_transforms(image_size)
    
    img = Image.open(image_path).convert('RGB')
    tensor = transform(img).unsqueeze(0).to(device)
    
    model.eval()
    with torch.no_grad():
        logits = model(tensor)
        probs = F.softmax(logits, dim=1)[0]
    
    pred_idx = probs.argmax().item()
    
    return {
        'emotion': labels[pred_idx],
        'confidence': probs[pred_idx].item(),
        'probabilities': {labels[i]: probs[i].item() for i in range(len(labels))},
    }


# Test on some validation images
print("\n🎭 Inference Demo — Predictions on validation samples:\n")

sample_indices = random.sample(range(len(val_dataset)), min(8, len(val_dataset)))

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

for i, idx in enumerate(sample_indices):
    ax = axes[i // 4, i % 4]
    
    image, true_label = val_dataset[idx]
    
    model.eval()
    with torch.no_grad():
        logits = model(image.unsqueeze(0).to(DEVICE))
        probs = F.softmax(logits, dim=1)[0]
    
    pred_idx = probs.argmax().item()
    pred_label = cfg.emotion_labels[pred_idx]
    true_name = cfg.emotion_labels[true_label]
    confidence = probs[pred_idx].item()
    
    # Display
    img_display = (image * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
    ax.imshow(img_display)
    
    color = 'green' if pred_idx == true_label else 'red'
    ax.set_title(f'Pred: {pred_label} ({confidence:.0%})\nTrue: {true_name}',
                 fontsize=10, color=color, fontweight='bold')
    ax.axis('off')

plt.suptitle('Inference Demo — Validation Samples', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(cfg.output_dir, 'inference_demo.png'), dpi=150, bbox_inches='tight')
plt.show()

## 12. Next Steps

After training, download these files from the supercomputer:

```
outputs/
├── best_model.pth          # PyTorch checkpoint (for fine-tuning)
├── emotion_model.onnx      # ONNX model (for production inference)
├── results.json            # Metrics & metadata
├── training_curves.png     # Loss/accuracy plots
├── confusion_matrix.png    # Per-class analysis
└── inference_demo.png      # Sample predictions
```

### To improve results:
1. **Train on RAF-DB**: Change `cfg.dataset = "rafdb"` — cleaner labels, higher ceiling
2. **Train on AffectNet**: Change `cfg.dataset = "affectnet7"` — largest dataset, best generalization
3. **Bigger backbone**: Try `cfg.backbone = "convnext_small.fb_in22k_ft_in1k"` or `"swin_tiny_patch4_window7_224.ms_in22k_ft_in1k"`
4. **Add landmark branch**: Integrate MediaPipe landmarks for geometric features
5. **Multi-dataset training**: Combine FER2013 + RAF-DB + AffectNet for maximum robustness

In [ ]:
print("\n" + "=" * 70)
print("✅ ALL DONE!")
print("=" * 70)
print(f"\n📦 Output directory: {os.path.abspath(cfg.output_dir)}")
print(f"\n🏆 Best Results:")
print(f"   Accuracy:     {best_acc:.4f}")
print(f"   Macro F1:     {best_uf1:.4f}")
print(f"   Best Epoch:   {best_epoch}")
print(f"\n📥 Download the outputs/ directory to your local machine.")
print(f"   Then run: python scripts/demo_webcam.py --model outputs/emotion_model.onnx")
print("=" * 70)

In [ ]:
# ==========================================
# ATTENTION EXPLAINABILITY
# ==========================================
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

def visualize_emotion_attention(model, image_tensor, original_image, true_label, emotion_labels):
    model.eval()
    with torch.no_grad():
        logits, attn_weights, shapes = model.extract_attention_maps(image_tensor.to(DEVICE))
        probs = F.softmax(logits, dim=1)[0]
        pred_label = torch.argmax(probs).item()
        
    attn = attn_weights[0] # (num_emotions, 1029)
    
    # We want to visualize the attention for the predicted emotion
    target_emotions = [pred_label]
    if true_label != pred_label:
        target_emotions.append(true_label)
        
    for emotion_idx in target_emotions:
        emotion_attn = attn[emotion_idx] # (1029,)
        
        # Split into scales
        len_s2 = shapes[0][0] * shapes[0][1]
        len_s3 = shapes[1][0] * shapes[1][1]
        len_s4 = shapes[2][0] * shapes[2][1]
        
        attn_s2 = emotion_attn[:len_s2].reshape(1, 1, shapes[0][0], shapes[0][1])
        attn_s3 = emotion_attn[len_s2:len_s2+len_s3].reshape(1, 1, shapes[1][0], shapes[1][1])
        attn_s4 = emotion_attn[len_s2+len_s3:].reshape(1, 1, shapes[2][0], shapes[2][1])
        
        # Upsample to original image size (224x224)
        attn_s2_up = F.interpolate(attn_s2, size=(224, 224), mode='bilinear', align_corners=False)
        attn_s3_up = F.interpolate(attn_s3, size=(224, 224), mode='bilinear', align_corners=False)
        attn_s4_up = F.interpolate(attn_s4, size=(224, 224), mode='bilinear', align_corners=False)
        
        # Average the multi-scale attention
        fused_attn = (attn_s2_up + attn_s3_up + attn_s4_up) / 3.0
        fused_attn = fused_attn.squeeze().cpu().numpy()
        
        # Normalize for visualization
        fused_attn = (fused_attn - fused_attn.min()) / (fused_attn.max() - fused_attn.min() + 1e-8)
        
        # Plot
        plt.figure(figsize=(10, 4))
        
        plt.subplot(1, 3, 1)
        plt.imshow(original_image)
        plt.title("Original Image")
        plt.axis('off')
        
        plt.subplot(1, 3, 2)
        plt.imshow(fused_attn, cmap='jet')
        plt.title(f"Attention Map: {emotion_labels[emotion_idx]}")
        plt.axis('off')
        
        plt.subplot(1, 3, 3)
        plt.imshow(original_image)
        plt.imshow(fused_attn, cmap='jet', alpha=0.5)
        plt.title(f"Overlay ({emotion_labels[emotion_idx]})")
        plt.axis('off')
        
        status = "Correct Prediction" if emotion_idx == true_label else "Prediction Error Analysis"
        plt.suptitle(f"{status} - Querying: {emotion_labels[emotion_idx].upper()}", fontweight='bold')
        plt.show()

print("✅ Explainability Visualization Function Ready!")
# To run this, pick an image from the dataloader:
# images, labels = next(iter(val_loader))
# original = images[0].permute(1, 2, 0).numpy()
# original = (original - original.min()) / (original.max() - original.min())
# visualize_emotion_attention(model, images[0].unsqueeze(0), original, labels[0].item(), cfg.emotion_labels)
